# IC4Net 22-Feature Prediction Notebook

This notebook is generated from `prediction_notebook_source.py`.
Upload it together with `ic4net_model.json` on the platform.


In [ ]:
from __future__ import annotations

"""
Paste this file's contents into the platform notebook as-is, or generate the
companion ipynb with `build_prediction_notebook.py`.

The notebook expects the exported `ic4net_model.json` to be uploaded in the
same working directory. It rebuilds the IC4Net 22-feature panel from only
`datasources["bar1m"]` and `datasources["financial"]`, applies the exported
ic3net-style residual cascade, and returns only `date`, `instrument`, `factor`.
"""


def main(datasources, start_date, end_date, model_path="ic4net_model.json"):
    import json
    import numpy as np
    import pandas as pd
    import dai
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    BUFFER_DAYS = 400
    EPS = 1e-12
    EXPECTED_MINUTES = 240
    N_GROUPS = 3
    BATCH_PRED = 50000
    FEATURE_COLS = [
        "1.11 VWAP Close Ratio Last30Min",
        "1.9 ExtremeHighReversal Last30Min",
        "1.7 DealRecovery Slope 30min",
        "1.2 M4",
        "1.4 lz 10",
        "1.1 Q3",
        "AmountMA20",
        "1.10 UpsideVolRatio Vol5DivVol20",
        "ATR5",
        "1.8 EarlyAfternoonRecovery VolumeWeighted VolatilityAdjusted",
        "PMCloseRange",
        "1.12 VolumeConcentration Last30Min PriceWeighted",
        "HF_GapII_SUM3D",
        "VRSI60",
        "FC_AssetImpairmentToRevenue_TTM",
        "AmountIR5",
        "Dratio",
        "VR_N",
        "EP_TTM",
        "CurrentAssetsTRate",
        "VEMA5",
        "SP1_TTM",
    ]

    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    query_start = (start_ts - pd.Timedelta(days=BUFFER_DAYS)).strftime("%Y-%m-%d %H:%M:%S")
    bar1m_table = datasources["bar1m"]
    financial_table = datasources["financial"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def winsor_mad(series, n=5):
        valid = series.dropna()
        if valid.empty:
            return series
        median = valid.median()
        mad = (valid - median).abs().median()
        if pd.isna(mad) or mad == 0:
            return series
        return series.clip(median - n * mad, median + n * mad)

    def zscore(series):
        valid = series.dropna()
        if valid.empty:
            return pd.Series(np.nan, index=series.index, dtype="float64")
        std = valid.std()
        centered = series - valid.mean()
        if pd.isna(std) or std == 0:
            return centered
        return centered / std

    def fill_cross_section(series):
        valid = series.dropna()
        if valid.empty:
            return pd.Series(0.0, index=series.index, dtype="float64")
        return series.fillna(valid.median())

    def safe_divide(left, right):
        left_num = pd.to_numeric(pd.Series(left), errors="coerce").astype("float64")
        right_num = pd.to_numeric(pd.Series(right), errors="coerce").replace(0.0, np.nan).astype("float64")
        return left_num / right_num

    def weighted_rolling_sum(series, weights, min_periods):
        return series.rolling(len(weights), min_periods=min_periods).apply(
            lambda y: float(np.sum(y * weights[-len(y):])),
            raw=True,
        )

    class SimpleScaler:
        def __init__(self, mean, scale):
            self.mean = np.asarray(mean, dtype=np.float32)
            self.scale = np.asarray(scale, dtype=np.float32)

        def transform(self, x):
            return (x - self.mean) / self.scale

    class FactorMLP(nn.Module):
        def __init__(self, input_dim, hidden_dims=None):
            super().__init__()
            if hidden_dims is None:
                hidden_dims = [64, 32, 16]
            layers = []
            prev_dim = input_dim
            for h in hidden_dims:
                layers.extend([nn.Linear(prev_dim, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(0.1)])
                prev_dim = h
            layers.append(nn.Linear(prev_dim, 1))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x).squeeze(-1)

    class ResidualHead(nn.Module):
        def __init__(self, input_dim, hidden_dims=None):
            super().__init__()
            if hidden_dims is None:
                hidden_dims = [48, 24, 12]
            layers = []
            prev_dim = input_dim
            for h in hidden_dims:
                layers.extend([nn.Linear(prev_dim, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(0.1)])
                prev_dim = h
            layers.append(nn.Linear(prev_dim, 1))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x).squeeze(-1)

    def load_models(json_path, map_location="cpu"):
        with open(json_path, "r", encoding="utf-8") as fh:
            payload = json.load(fh)

        def _deserialize(tensors):
            state_dict = {}
            for key, meta in tensors.items():
                tensor = torch.tensor(meta["data"], dtype=getattr(torch, meta["dtype"]))
                state_dict[key] = tensor.reshape(meta["shape"]).to(map_location)
            return state_dict

        models = []
        models.append(FactorMLP(**payload["model_cfg"]).to(map_location))
        models[0].load_state_dict(_deserialize(payload["models"]["net0"]))
        models[0].eval()
        for i in range(1, payload["n_groups"]):
            model = ResidualHead(**payload["residual_cfg"]).to(map_location)
            model.load_state_dict(_deserialize(payload["models"][f"net{i}"]))
            model.eval()
            models.append(model)
        scalers = [
            SimpleScaler(payload["scalers"][str(i)]["mean"], payload["scalers"][str(i)]["scale"])
            for i in range(payload["n_groups"])
        ]
        return models, scalers, payload["feature_cols"], payload["n_groups"]

    def run_cascade_prediction(models, scalers, test_df, test_raw, device, n_groups=N_GROUPS, batch_pred=BATCH_PRED):
        for model in models:
            model.eval()
        n_test = len(test_raw)
        chains = np.zeros((n_test, n_groups), dtype=np.float32)

        for start in range(0, n_test, batch_pred):
            end = min(start + batch_pred, n_test)
            batch_raw = test_raw[start:end]
            x0 = torch.tensor(scalers[0].transform(batch_raw), dtype=torch.float32).to(device)
            with torch.no_grad():
                accum = models[0](x0)
            chains[start:end, 0] = accum.cpu().numpy()
            for k in range(1, n_groups):
                xk = torch.tensor(scalers[k].transform(batch_raw), dtype=torch.float32).to(device)
                with torch.no_grad():
                    accum = accum + F.softplus(models[k](xk))
                chains[start:end, k] = accum.cpu().numpy()

        scored = test_df[["date", "instrument"]].copy()
        scored["factor"] = 0.0

        def _cascade(group):
            n = len(group)
            gi = group.index.values
            remaining = np.ones(n, dtype=bool)
            for g in range(n_groups - 1):
                n_rem = remaining.sum()
                if n_rem == 0:
                    break
                pool_local = np.where(remaining)[0]
                pool_global = gi[pool_local]
                score_g = chains[pool_global, g]
                order = np.argsort(score_g)
                frac = 1.0 / (n_groups - g)
                n_take = max(1, int(n_rem * frac))
                take_local = pool_local[order[:n_take]]
                take_global = gi[take_local]
                group.loc[group.index[take_local], "factor"] = chains[take_global, g]
                remaining[take_local] = False
            if remaining.sum() > 0:
                last_local = np.where(remaining)[0]
                last_global = gi[last_local]
                group.loc[group.index[last_local], "factor"] = chains[last_global, n_groups - 1]
            return group

        return scored.groupby("date", group_keys=False).apply(_cascade).reset_index(drop=True)

    technical_sql = f"""
    WITH minute_base AS (
        SELECT
            date_trunc('day', date)::DATE AS trading_day,
            date,
            instrument,
            CAST(strftime(date, '%H') AS INTEGER) AS hour_of_day,
            CAST(strftime(date, '%M') AS INTEGER) AS minute_of_hour,
            CAST(open AS DOUBLE) AS open,
            CAST(high AS DOUBLE) AS high,
            CAST(low AS DOUBLE) AS low,
            CAST(close AS DOUBLE) AS close,
            CAST(volume AS DOUBLE) AS volume,
            CAST(amount AS DOUBLE) AS amount,
            CAST(deal_number AS DOUBLE) AS deal_number,
            ROW_NUMBER() OVER (
                PARTITION BY instrument, date_trunc('day', date)::DATE
                ORDER BY date
            ) - 1 AS bar_idx,
            COUNT(*) OVER (
                PARTITION BY instrument, date_trunc('day', date)::DATE
            ) AS bar_count,
            LAG(CAST(close AS DOUBLE)) OVER (
                PARTITION BY instrument, date_trunc('day', date)::DATE
                ORDER BY date
            ) AS prev_close_in_day
        FROM {bar1m_table}
        WHERE close > 0
    ),
    minute_enriched AS (
        SELECT
            *,
            CASE
                WHEN prev_close_in_day > 0 AND close > 0 THEN LN(close / prev_close_in_day)
                ELSE NULL
            END AS minute_ret,
            SUM(COALESCE(deal_number, 0.0)) OVER (
                PARTITION BY trading_day, instrument
            ) AS total_deals_day,
            SUM(COALESCE(deal_number, 0.0)) OVER (
                PARTITION BY trading_day, instrument
                ORDER BY COALESCE(volume, 0.0) DESC, date
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS cum_deals_by_volume,
            SUM(COALESCE(deal_number, 0.0)) OVER (
                PARTITION BY trading_day, instrument
                ORDER BY COALESCE(volume, 0.0) DESC, date
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ) AS cum_deals_prev
        FROM minute_base
    )
    SELECT
        trading_day AS date,
        instrument,
        SUM(volume) AS daily_volume,
        SUM(amount) AS daily_amount,
        SUM(COALESCE(deal_number, 0.0)) AS daily_deals,
        COUNT(*) AS minute_count,
        ARG_MIN(open, date) AS daily_open,
        ARG_MAX(close, date) AS close_last,
        MAX(high) AS daily_high,
        MIN(low) AS daily_low,
        MAX(close) FILTER (WHERE bar_idx = 9) AS close_bar10,
        MAX(close) FILTER (WHERE bar_idx = bar_count - 11) AS close_prev11,
        MAX(close) FILTER (WHERE bar_idx = 180) AS close_bar181,
        MAX(high) FILTER (
            WHERE hour_of_day > 14 OR (hour_of_day = 14 AND minute_of_hour >= 30)
        ) AS pm_high,
        MIN(low) FILTER (
            WHERE hour_of_day > 14 OR (hour_of_day = 14 AND minute_of_hour >= 30)
        ) AS pm_low,
        SUM(CASE WHEN minute_ret IS NOT NULL THEN POWER(minute_ret, 2) ELSE NULL END) AS ret_sq_sum,
        SUM(CASE WHEN minute_ret < 0 THEN POWER(minute_ret, 2) ELSE 0 END) AS neg_ret_sq_sum,
        SUM(CASE WHEN hour_of_day = 9 AND minute_of_hour >= 31 THEN volume ELSE 0 END)
            + SUM(CASE WHEN hour_of_day = 10 AND minute_of_hour = 0 THEN volume ELSE 0 END) AS morning30_volume,
        SUM(CASE WHEN hour_of_day = 13 AND minute_of_hour >= 1 AND minute_of_hour <= 30 THEN volume ELSE 0 END) AS afternoon30_volume,
        SUM(
            CASE
                WHEN (hour_of_day = 13 OR (hour_of_day = 14 AND minute_of_hour = 0)) AND minute_ret > 0
                THEN minute_ret * volume
                ELSE 0
            END
        ) AS early_pos_retvol,
        SUM(
            CASE
                WHEN (hour_of_day = 13 OR (hour_of_day = 14 AND minute_of_hour = 0)) AND minute_ret < 0
                THEN ABS(minute_ret) * volume
                ELSE 0
            END
        ) AS early_neg_retvol,
        STDDEV_SAMP(
            CASE
                WHEN hour_of_day = 13 OR (hour_of_day = 14 AND minute_of_hour = 0)
                THEN minute_ret
                ELSE NULL
            END
        ) AS early_sigma,
        AVG(
            CASE
                WHEN hour_of_day = 14 AND minute_of_hour >= 30 AND minute_of_hour < 45
                THEN deal_number
                ELSE NULL
            END
        ) AS deal_pre_mean,
        AVG(
            CASE
                WHEN hour_of_day = 14 AND minute_of_hour >= 45
                THEN deal_number
                WHEN hour_of_day = 15 AND minute_of_hour = 0
                THEN deal_number
                ELSE NULL
            END
        ) AS deal_post_mean,
        AVG(
            CASE
                WHEN hour_of_day > 14 OR (hour_of_day = 14 AND minute_of_hour >= 30)
                THEN POWER(CASE WHEN minute_ret > 0 THEN minute_ret ELSE 0 END, 2)
                ELSE NULL
            END
        ) AS tail30_pos_sq_mean,
        AVG(
            CASE
                WHEN hour_of_day > 14 OR (hour_of_day = 14 AND minute_of_hour >= 30)
                THEN POWER(CASE WHEN minute_ret < 0 THEN minute_ret ELSE 0 END, 2)
                ELSE NULL
            END
        ) AS tail30_neg_sq_mean,
        ARG_MIN(open, date) FILTER (
            WHERE hour_of_day > 14 OR (hour_of_day = 14 AND minute_of_hour >= 30)
        ) AS tail30_open,
        SUM(
            CASE
                WHEN hour_of_day > 14 OR (hour_of_day = 14 AND minute_of_hour >= 30)
                THEN volume
                ELSE 0
            END
        ) AS tail30_volume,
        SUM(
            CASE
                WHEN hour_of_day > 14 OR (hour_of_day = 14 AND minute_of_hour >= 30)
                THEN amount
                ELSE 0
            END
        ) AS tail30_amount,
        SUM(
            CASE
                WHEN hour_of_day = 14 AND minute_of_hour >= 30 AND minute_of_hour < 45
                THEN volume
                ELSE 0
            END
        ) AS prev15_vol,
        SUM(
            CASE
                WHEN hour_of_day = 14 AND minute_of_hour >= 45
                THEN volume
                WHEN hour_of_day = 15 AND minute_of_hour = 0
                THEN volume
                ELSE 0
            END
        ) AS last15_vol,
        ARG_MIN(close, date) FILTER (
            WHERE hour_of_day = 14 AND minute_of_hour >= 45
               OR hour_of_day = 15 AND minute_of_hour = 0
        ) AS p_1445,
        SUM(
            CASE
                WHEN hour_of_day = 14 AND minute_of_hour >= 45 AND high > low
                THEN volume * ((close - low) / (high - low))
                WHEN hour_of_day = 15 AND minute_of_hour = 0 AND high > low
                THEN volume * ((close - low) / (high - low))
                ELSE 0
            END
        ) AS last15_weighted_vol,
        SUM(
            CASE
                WHEN COALESCE(cum_deals_prev, 0.0) < 0.445 * total_deals_day
                THEN volume
                ELSE 0
            END
        ) AS q3_subset_volume,
        SUM(
            CASE
                WHEN COALESCE(cum_deals_prev, 0.0) < 0.445 * total_deals_day
                THEN amount
                ELSE 0
            END
        ) AS q3_subset_amount
    FROM minute_enriched
    GROUP BY trading_day, instrument
    """

    technical = dai.query(
        technical_sql,
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()
    if technical.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    technical["date"] = pd.to_datetime(technical["date"])
    technical["instrument"] = technical["instrument"].astype(str)
    numeric_cols = [c for c in technical.columns if c not in ["date", "instrument"]]
    if numeric_cols:
        technical[numeric_cols] = technical[numeric_cols].apply(pd.to_numeric, errors="coerce").astype("float64")
    technical.sort_values(["instrument", "date"], inplace=True)
    technical.reset_index(drop=True, inplace=True)

    grouped = technical.groupby("instrument", sort=False)
    prev_close = grouped["close_last"].shift(1)
    technical["daily_simple_ret"] = technical["close_last"] / prev_close - 1.0

    overnight_gap = np.log((technical["daily_open"] + EPS) / (prev_close + EPS))
    technical["HF_GapII"] = np.sign(technical["close_last"] / technical["close_prev11"] - 1.0) / (
        1.0 + (np.log((technical["close_bar10"] + EPS) / (technical["daily_open"] + EPS)) - overnight_gap).abs()
    )
    technical["HF_GapII_SUM3D"] = grouped["HF_GapII"].transform(lambda s: s.rolling(3, min_periods=2).sum())

    ma5_amt = grouped["daily_amount"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    std5_amt = grouped["daily_amount"].transform(lambda s: s.rolling(5, min_periods=3).std())
    ma20_20_amt = grouped["daily_amount"].transform(
        lambda s: s.rolling(20, min_periods=10).mean().rolling(20, min_periods=10).mean()
    )
    technical["AmountIR5"] = safe_divide(ma5_amt, std5_amt)
    technical["AmountMA20"] = safe_divide(technical["daily_amount"], ma20_20_amt)

    vol_gap = grouped["daily_volume"].diff()
    gap_pos = vol_gap.clip(lower=0.0)
    gap_abs = vol_gap.abs()
    vrsi_weights = np.array([2.0] + [1.0] * 59, dtype="float64")[::-1] / 61.0
    technical["vrsi_pos_w"] = np.nan
    technical["vrsi_abs_w"] = np.nan
    for _, idx in grouped.groups.items():
        technical.loc[idx, "vrsi_pos_w"] = weighted_rolling_sum(gap_pos.loc[idx].astype("float64"), vrsi_weights, 60).to_numpy()
        technical.loc[idx, "vrsi_abs_w"] = weighted_rolling_sum(gap_abs.loc[idx].astype("float64"), vrsi_weights, 60).to_numpy()
    technical["VRSI60"] = safe_divide(technical["vrsi_pos_w"], technical["vrsi_abs_w"])

    technical["vrn_daily"] = safe_divide(technical["morning30_volume"], technical["afternoon30_volume"])
    technical["VR_N"] = grouped["vrn_daily"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    technical["dratio_raw"] = safe_divide(technical["neg_ret_sq_sum"], technical["ret_sq_sum"])
    technical["Dratio"] = grouped["dratio_raw"].transform(lambda s: s.rolling(5, min_periods=1).mean())

    technical["pmclose_raw"] = -safe_divide(technical["close_last"] - technical["pm_low"], technical["pm_high"] - technical["pm_low"])
    pm_wins = technical.groupby("date", sort=False)["pmclose_raw"].transform(winsor_mad)
    technical["PMCloseRange"] = pm_wins.groupby(technical["date"], sort=False).rank(pct=True)

    tr_components = pd.concat(
        [
            (technical["daily_high"] - technical["daily_low"]).rename("hl"),
            (technical["daily_high"] - prev_close).abs().rename("hc"),
            (technical["daily_low"] - prev_close).abs().rename("lc"),
        ],
        axis=1,
    )
    technical["TR"] = tr_components.max(axis=1)
    technical["ATR5"] = -grouped["TR"].transform(lambda s: s.rolling(5, min_periods=3).mean())

    vema_weights = np.array([2.0] + [1.0] * 4, dtype="float64")[::-1] / 6.0
    technical["VEMA5"] = -grouped["daily_volume"].transform(
        lambda s: s.rolling(5, min_periods=5).apply(lambda y: float(np.sum(y * vema_weights)), raw=True)
    )

    technical["1.4 lz 10"] = safe_divide(
        grouped["daily_amount"].transform(lambda s: s.rolling(10, min_periods=5).mean()),
        grouped["daily_amount"].transform(lambda s: s.rolling(20, min_periods=10).std()),
    )

    technical["1.2 M4_raw"] = np.where(
        (technical["minute_count"] >= EXPECTED_MINUTES) & (technical["close_bar181"] > 0),
        technical["close_last"] / technical["close_bar181"] - 1.0,
        np.nan,
    )
    technical["1.2 M4"] = grouped["1.2 M4_raw"].transform(lambda s: s.rolling(5, min_periods=5).mean())

    technical["1.1 Q3_raw"] = np.where(
        (technical["daily_deals"] > 0) & (technical["daily_volume"] > 0) & (technical["q3_subset_volume"] > 0),
        (technical["daily_amount"] / technical["daily_volume"]) / (technical["q3_subset_amount"] / technical["q3_subset_volume"]),
        np.nan,
    )
    technical["1.1 Q3"] = grouped["1.1 Q3_raw"].transform(lambda s: s.rolling(4, min_periods=4).mean())

    vol5 = grouped["daily_simple_ret"].transform(lambda s: s.rolling(5, min_periods=3).std())
    vol20 = grouped["daily_simple_ret"].transform(lambda s: s.rolling(20, min_periods=10).std())
    upside_ratio_raw = safe_divide(technical["tail30_pos_sq_mean"], technical["tail30_neg_sq_mean"])
    technical["1.10 UpsideVolRatio Vol5DivVol20"] = -upside_ratio_raw * safe_divide(vol5, vol20)

    technical["1.8 EarlyAfternoonRecovery VolumeWeighted VolatilityAdjusted"] = safe_divide(
        safe_divide(technical["early_pos_retvol"], technical["early_neg_retvol"]),
        technical["early_sigma"],
    )

    technical["1.7 DealRecovery Slope 30min_raw"] = technical["deal_pre_mean"] - technical["deal_post_mean"]
    drs_wins = technical.groupby("date", sort=False)["1.7 DealRecovery Slope 30min_raw"].transform(winsor_mad)
    technical["1.7 DealRecovery Slope 30min"] = drs_wins.groupby(technical["date"], sort=False).rank(pct=True)

    technical["1.9 ExtremeHighReversal Last30Min"] = np.where(
        technical["tail30_open"] > 0,
        -(technical["pm_high"] / technical["tail30_open"] - 1.0),
        np.nan,
    )
    technical["1.11 VWAP Close Ratio Last30Min"] = safe_divide(
        safe_divide(technical["tail30_amount"], technical["tail30_volume"]),
        technical["close_last"],
    )
    technical["1.12 VolumeConcentration Last30Min PriceWeighted"] = -safe_divide(
        technical["last15_weighted_vol"],
        technical["prev15_vol"],
    )

    financial_sql = f"""
    SELECT
        CAST(date AS DATE) AS date,
        instrument,
        category,
        shift,
        CAST(share_capital AS DOUBLE) AS share_capital,
        CAST(total_current_assets AS DOUBLE) AS total_current_assets,
        CAST(total_operating_revenue AS DOUBLE) AS total_operating_revenue,
        CAST(asset_impairment_loss AS DOUBLE) AS asset_impairment_loss,
        CAST(credit_impairment_loss AS DOUBLE) AS credit_impairment_loss,
        CAST(net_profit_to_parent_shareholders AS DOUBLE) AS net_profit_to_parent_shareholders
    FROM {financial_table}
    WHERE (category = 'lf' AND shift IN (0, 4))
       OR (category = 'ttm' AND shift = 0)
    """

    financial = dai.query(
        financial_sql,
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()
    financial["date"] = pd.to_datetime(financial["date"])
    financial["instrument"] = financial["instrument"].astype(str)
    for col in [
        "share_capital",
        "total_current_assets",
        "total_operating_revenue",
        "asset_impairment_loss",
        "credit_impairment_loss",
        "net_profit_to_parent_shareholders",
    ]:
        financial[col] = pd.to_numeric(financial[col], errors="coerce").astype("float64")

    lf0 = financial.loc[(financial["category"] == "lf") & (financial["shift"] == 0), [
        "date", "instrument", "share_capital", "total_current_assets"
    ]].drop_duplicates(["date", "instrument"], keep="last").rename(
        columns={"share_capital": "share_capital_lf0", "total_current_assets": "total_current_assets_lf0"}
    )
    lf4 = financial.loc[(financial["category"] == "lf") & (financial["shift"] == 4), [
        "date", "instrument", "total_current_assets"
    ]].drop_duplicates(["date", "instrument"], keep="last").rename(
        columns={"total_current_assets": "total_current_assets_lf4"}
    )
    ttm0 = financial.loc[(financial["category"] == "ttm") & (financial["shift"] == 0), [
        "date",
        "instrument",
        "total_operating_revenue",
        "asset_impairment_loss",
        "credit_impairment_loss",
        "net_profit_to_parent_shareholders",
    ]].drop_duplicates(["date", "instrument"], keep="last").rename(
        columns={
            "total_operating_revenue": "total_operating_revenue_ttm0",
            "asset_impairment_loss": "asset_impairment_loss_ttm0",
            "credit_impairment_loss": "credit_impairment_loss_ttm0",
            "net_profit_to_parent_shareholders": "net_profit_to_parent_shareholders_ttm0",
        }
    )

    frame = technical.merge(lf0, on=["date", "instrument"], how="left")
    frame = frame.merge(lf4, on=["date", "instrument"], how="left")
    frame = frame.merge(ttm0, on=["date", "instrument"], how="left")

    market_value = frame["share_capital_lf0"] * frame["close_last"]
    frame["SP1_TTM"] = safe_divide(frame["total_operating_revenue_ttm0"], market_value)
    frame["EP_TTM"] = safe_divide(frame["net_profit_to_parent_shareholders_ttm0"], market_value)
    frame["CurrentAssetsTRate"] = safe_divide(
        2.0 * frame["total_operating_revenue_ttm0"],
        frame["total_current_assets_lf0"] + frame["total_current_assets_lf4"],
    )
    frame["FC_AssetImpairmentToRevenue_TTM"] = 100.0 * safe_divide(
        frame["asset_impairment_loss_ttm0"] + frame["credit_impairment_loss_ttm0"],
        frame["total_operating_revenue_ttm0"],
    )

    frame = frame.loc[frame["date"].between(start_ts, end_ts)].copy()
    if frame.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    for col in FEATURE_COLS:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        frame[col] = frame.groupby("date", sort=False)[col].transform(winsor_mad)
        frame[col] = frame.groupby("date", sort=False)[col].transform(zscore)
        frame[col] = frame.groupby("date", sort=False)[col].transform(fill_cross_section).fillna(0.0)

    models, scalers, model_feature_cols, model_n_groups = load_models(model_path, map_location=device)
    test_df = frame[["date", "instrument"] + model_feature_cols].sort_values(["date", "instrument"]).reset_index(drop=True)
    test_raw = test_df[model_feature_cols].to_numpy(dtype=np.float64)
    pred = run_cascade_prediction(models, scalers, test_df, test_raw, device, n_groups=model_n_groups)
    pred.sort_values(["date", "instrument"], inplace=True)
    pred.reset_index(drop=True, inplace=True)
    return pred[["date", "instrument", "factor"]]
